In [1]:
from ultralytics import YOLO

import torch
import torchvision.transforms as transforms

from torchvision.models import resnet18
import torchvision.models as models
from PIL import Image

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from tqdm import tqdm
import time

In [2]:
YOLO_MODEL = YOLO("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\runs\\detect\\runs\\YOLOv8_baseline\\weights\\best.pt")

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = cnn.fc.in_features
cnn.fc = torch.nn.Sequential(
    torch.nn.Linear(num_features, 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(128, 2)
)

cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\best_chicken_cnn.pth"))

cnn.to(DEVICE)

cnn.eval()

C:\Users\klanz\AppData\Local\Temp\ipykernel_9260\2231296117.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\M

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [5]:
def classify_crop(crop):

    image = Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB))

    tensor = transform(image)

    tensor = tensor.unsqueeze(0)

    tensor = tensor.to(DEVICE)

    with torch.no_grad():

        prediction = cnn(tensor)

        prediction = prediction.argmax(1).item()

    return prediction

In [6]:
def door_decision(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    cnn_classes = [p["cnn_prediction"] for p in predictions]

    if 1 in cnn_classes:
        return "CLOSE"

    if 0 in cnn_classes:
        return "OPEN"

    return "CLOSE"

In [7]:
def door_decision_yolo(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    classes = [p["class"] for p in predictions]

    if 1 in classes:
        return "CLOSE"

    if 0 in classes:
        return "OPEN"

    return "CLOSE"

In [8]:
CLASS_NAMES = {
    0: "chicken",
    1: "not_chicken"
}

In [9]:
def run_yolo(image_path, conf=0.5):

    result = YOLO_MODEL.predict(
        source=str(image_path),
        conf=conf,
        verbose=False
    )[0]

    predictions = []

    for box in result.boxes:

        cls = int(box.cls.item())

        confidence = float(box.conf.item())

        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]

        predictions.append({

            "class": cls,
            "class_name": CLASS_NAMES[cls],
            "confidence": confidence,
            "bbox": [int(x1), int(y1), int(x2), int(y2)]

        })

    return predictions

In [10]:
def run_pipeline(image_path, conf=0.5):

    detections = run_yolo(image_path, conf)

    image = cv2.imread(str(image_path))

    final_predictions = []

    for det in detections:

        x1, y1, x2, y2 = det["bbox"]

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        cnn_prediction = classify_crop(crop)

        final_predictions.append({

            "yolo_prediction": det["class"],

            "cnn_prediction": cnn_prediction,

            "confidence": det["confidence"],

            "bbox": det["bbox"]

        })

    return final_predictions

In [11]:
def load_ground_truth(label_path):

    if not Path(label_path).exists():
        return []

    gt=[]

    with open(label_path) as f:

        for line in f:

            line=line.strip()

            if line=="":

                continue

            cls=int(line.split()[0])

            gt.append(cls)

    return gt

In [12]:
def ground_truth_decision(gt_classes):

    if 1 in gt_classes:
        return "CLOSE"

    if 0 in gt_classes:
        return "OPEN"

    return "CLOSE"

In [13]:
test_images = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset\\test\\images").glob("*"))

results = []

In [14]:
for image_path in test_images:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")

    gt = load_ground_truth(label_path)

    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()

    yolo_predictions = run_yolo(image_path)

    yolo_time = (time.perf_counter()-start)*1000

    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()

    pipeline_predictions = run_pipeline(image_path)

    pipeline_time = (time.perf_counter()-start)*1000

    pipeline_decision = door_decision(pipeline_predictions)

    results.append({

        "image": image_path.name,

        "ground_truth": gt_decision,

        "yolo": yolo_decision,

        "pipeline": pipeline_decision,

        "yolo_time_ms": yolo_time,

        "pipeline_time_ms": pipeline_time,

        "objects_gt": gt,

        "objects_yolo": [x["class"] for x in yolo_predictions],

        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [15]:
df = pd.DataFrame(results)

df.head(20)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,413.8674,62.8669,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,13.5343,17.1554,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,12.9485,25.3116,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,13.2750,16.6295,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,12.8263,28.5425,"[0, 0, 0]","[0, 0, 0, 0]","[0, 0, 0, 1]"
5,1085.jpeg,OPEN,OPEN,CLOSE,13.2112,17.3289,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,12.8101,16.3641,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,12.8192,16.6907,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,11.5544,16.7749,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,12.1130,20.3823,"[0, 0, 0]","[0, 0]","[0, 0]"


In [16]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

import numpy as np

In [17]:
decision_map = {

    "OPEN":1,

    "CLOSE":0

}

In [18]:
def evaluate_system(df, prediction_column, time_column):

    gt = df["ground_truth"].map(decision_map)

    pred = df[prediction_column].map(decision_map)

    accuracy = accuracy_score(gt, pred)

    precision = precision_score(
        gt,
        pred,
        zero_division=0
    )

    recall = recall_score(
        gt,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        gt,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(gt, pred)

    tn, fp, fn, tp = cm.ravel()

    avg_time = df[time_column].mean()

    print("="*50)

    print(prediction_column)

    print("="*50)

    print(f"Accuracy : {accuracy:.4f}")

    print(f"Precision: {precision:.4f}")

    print(f"Recall   : {recall:.4f}")

    print(f"F1-score : {f1:.4f}")

    print()

    print(f"TP : {tp}")

    print(f"FP : {fp}")

    print(f"TN : {tn}")

    print(f"FN : {fn}")

    print()

    print(f"Średni czas: {avg_time:.2f} ms")

    return {

        "Accuracy":accuracy,

        "Precision":precision,

        "Recall":recall,

        "F1":f1,

        "TP":tp,

        "FP":fp,

        "TN":tn,

        "FN":fn,

        "Time":avg_time

    }

In [19]:
yolo_results = evaluate_system(

    df,

    "yolo",

    "yolo_time_ms"

)

yolo
Accuracy : 0.9886
Precision: 0.9958
Recall   : 0.9836
F1-score : 0.9897

TP : 479
FP : 2
TN : 389
FN : 8

Średni czas: 14.29 ms


In [20]:
pipeline_results = evaluate_system(

    df,

    "pipeline",

    "pipeline_time_ms"

)

pipeline
Accuracy : 0.9841
Precision: 0.9958
Recall   : 0.9754
F1-score : 0.9855

TP : 475
FP : 2
TN : 389
FN : 12

Średni czas: 21.38 ms


In [21]:
comparison = pd.DataFrame(

    [

        yolo_results,

        pipeline_results

    ],

    index=[

        "YOLO",

        "YOLO + CNN"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.988610,0.995842,0.983573,0.989669,479,2,389,8,14.287762
YOLO + CNN,0.984055,0.995807,0.975359,0.985477,475,2,389,12,21.382157


In [22]:
dangerous_yolo = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN")

]

In [23]:
dangerous_pipeline = df[

    (df["ground_truth"]=="CLOSE") &

    (df["pipeline"]=="OPEN")

]

In [24]:
print()

print("Krytyczne błędy")

print("----------------")

print("YOLO:",len(dangerous_yolo))

print("YOLO+CNN:",len(dangerous_pipeline))


Krytyczne błędy
----------------
YOLO: 2
YOLO+CNN: 2


In [25]:
improved = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN") &

    (df["pipeline"]=="CLOSE")

]

improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,9.4449,13.2832,[1],[0],[1]


In [26]:
empty_images = 0
false_detections = 0

for row in results:

    if len(row["objects_gt"]) == 0:

        empty_images += 1

        if len(row["objects_yolo"]) > 0:
            false_detections += 1

print("Puste obrazy:",empty_images)
print("Fałszywe detekcje:",false_detections)

if empty_images>0:

    print(
        "Odsetek:",
        false_detections/empty_images
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [27]:
fp_images=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["pipeline"]=="OPEN":

        fp_images.append(row["image"])

print("False Positive:",len(fp_images))

fp_images

False Positive: 2


['Image-51-eb7770.jpg', 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [28]:
fn_images=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["pipeline"]=="CLOSE":

        fn_images.append(row["image"])

print("False Negative:",len(fn_images))

fn_images

False Negative: 12


['1054.jpeg',
 '1085.jpeg',
 '561.jpeg',
 'neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-9Q37DlviKYpHMGPjddECLgAAAA.jpeg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-FbBPwSbJW6mmwK-7AlqNRAHaEK.jpeg',
 'OIP-FY33flUTFsva0xhQAxhZwgHaEK.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg',
 'OIP-tX7YgAqzRxQxMYkGe_FCtwHaFj.jpeg',
 'OIP-z_Y26rFvbazALq6sfXjgxwHaE8.jpeg']

In [29]:
fp_yolo=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["yolo"]=="OPEN":

        fp_yolo.append(row["image"])

len(fp_yolo)

fp_yolo

['Image-84-bd2f1b.jpg', 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [30]:
fn_yolo=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["yolo"]=="CLOSE":

        fn_yolo.append(row["image"])

len(fn_yolo)

fn_yolo

['neg_poultry__poultry_222_jpg.rf.g0AgDML3T5PV9uIHp0Xd.jpg',
 'neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-aoDZ9NQTzSJ5RInD_vwARgHaFj.jpeg',
 'OIP-FY33flUTFsva0xhQAxhZwgHaEK.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg',
 'OIP-RtkErMx6_ECG2NR4KTQiTQHaE8.jpeg',
 'OIP-tX7YgAqzRxQxMYkGe_FCtwHaFj.jpeg',
 'OIP-z_Y26rFvbazALq6sfXjgxwHaE8.jpeg']

In [31]:

OUTPUT = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset_experiments")



In [32]:
image_path_dark = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\dark\\images").glob("*"))
image_path_night = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\night\\images").glob("*"))
image_path_occlusion = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\occlusion\\images").glob("*"))
image_path_motion_blur = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\motion_blur\\images").glob("*"))

results_dark = []
results_night = []
results_occlusion = []
results_motion_blur = []


In [33]:
for image_path in image_path_dark:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_dark.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [34]:
for image_path in image_path_night:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_night.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [35]:
for image_path in image_path_occlusion:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_occlusion.append({
        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [36]:
for image_path in image_path_motion_blur:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_motion_blur.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [37]:
df_dark = pd.DataFrame(results_dark)
df_night = pd.DataFrame(results_night)
df_occlusion = pd.DataFrame(results_occlusion)
df_motion_blur = pd.DataFrame(results_motion_blur)

df_dark.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,74.5082,79.5472,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,52.7734,68.5193,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,51.2455,35.8652,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,13.2750,15.5457,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,OPEN,11.0230,22.3491,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"
5,1085.jpeg,OPEN,OPEN,OPEN,11.0372,15.7219,[0],[0],[0]
6,109.jpeg,OPEN,OPEN,OPEN,11.2626,16.4110,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,12.1057,15.6507,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,13.5238,17.9310,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,10.6526,18.9150,"[0, 0, 0]","[0, 0]","[0, 0]"


In [38]:
df_night.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,16.0046,16.5404,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,15.0876,35.4770,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,14.4032,20.2577,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,10.8986,16.0418,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,12.2344,28.5565,"[0, 0, 0]","[0, 0, 0, 0]","[0, 1, 0, 0]"
5,1085.jpeg,OPEN,OPEN,OPEN,11.8547,16.9054,[0],[0],[0]
6,109.jpeg,OPEN,OPEN,OPEN,12.3584,17.7445,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,11.2647,17.7933,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,11.5226,16.8089,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,11.9494,20.7020,"[0, 0, 0]","[0, 0]","[0, 0]"


In [39]:
df_occlusion.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,21.4886,11.5311,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,10.9552,16.1226,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,11.6898,20.1852,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,11.6573,16.2145,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,11.2046,26.2639,"[0, 0, 0]","[0, 0, 0, 0]","[0, 0, 1, 0]"
5,1085.jpeg,OPEN,OPEN,CLOSE,10.9534,19.6758,[0],"[0, 0]","[1, 1]"
6,109.jpeg,OPEN,OPEN,OPEN,10.9211,14.8537,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,14.4087,15.3804,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,11.0880,15.0996,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,9.6415,17.6119,"[0, 0, 0]","[0, 0]","[0, 1]"


In [40]:
df_motion_blur.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,13.7346,17.9025,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,CLOSE,11.8988,17.5495,[0],[0],[1]
2,1048.jpeg,OPEN,CLOSE,CLOSE,10.0921,11.1597,[0],[],[]
3,1053.jpeg,OPEN,OPEN,OPEN,12.6334,17.4381,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,CLOSE,CLOSE,11.3653,16.1327,"[0, 0, 0]",[1],[1]
5,1085.jpeg,OPEN,OPEN,CLOSE,11.7560,16.3588,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,11.6442,16.5338,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,11.6281,17.2675,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,11.5362,18.2681,[0],[0],[0]
9,1133.jpeg,OPEN,CLOSE,CLOSE,10.0057,11.0120,"[0, 0, 0]",[],[]


In [41]:
yolo_results1 = evaluate_system(
    df_dark,
    "yolo",
    "yolo_time_ms"
)
pipeline_results1 = evaluate_system(
    df_dark,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9829
Precision: 0.9876
Recall   : 0.9815
F1-score : 0.9846

TP : 478
FP : 6
TN : 385
FN : 9

Średni czas: 12.60 ms
pipeline
Accuracy : 0.9692
Precision: 0.9618
Recall   : 0.9836
F1-score : 0.9726

TP : 479
FP : 19
TN : 372
FN : 8

Średni czas: 20.28 ms


In [42]:
yolo_results2 = evaluate_system(
    df_night,
    "yolo",
    "yolo_time_ms"
)
pipeline_results2 = evaluate_system(
    df_night,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9658
Precision: 0.9831
Recall   : 0.9548
F1-score : 0.9688

TP : 465
FP : 8
TN : 383
FN : 22

Średni czas: 11.63 ms
pipeline
Accuracy : 0.9226
Precision: 0.9084
Recall   : 0.9569
F1-score : 0.9320

TP : 466
FP : 47
TN : 344
FN : 21

Średni czas: 18.47 ms


In [43]:
yolo_results3 = evaluate_system(
    df_occlusion,
    "yolo",
    "yolo_time_ms"
)
pipeline_results3 = evaluate_system(
    df_occlusion,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9590
Precision: 0.9913
Recall   : 0.9343
F1-score : 0.9619

TP : 455
FP : 4
TN : 387
FN : 32

Średni czas: 12.14 ms
pipeline
Accuracy : 0.9419
Precision: 0.9802
Recall   : 0.9138
F1-score : 0.9458

TP : 445
FP : 9
TN : 382
FN : 42

Średni czas: 19.55 ms


In [44]:
yolo_results4 = evaluate_system(
    df_motion_blur,
    "yolo",
    "yolo_time_ms"
)
pipeline_results4 = evaluate_system(
    df_motion_blur,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.8189
Precision: 0.9227
Recall   : 0.7351
F1-score : 0.8183

TP : 358
FP : 30
TN : 361
FN : 129

Średni czas: 11.63 ms
pipeline
Accuracy : 0.7335
Precision: 0.9847
Recall   : 0.5277
F1-score : 0.6872

TP : 257
FP : 4
TN : 387
FN : 230

Średni czas: 16.76 ms


In [45]:
comparison = pd.DataFrame(
    [
        yolo_results1,
        pipeline_results1
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.982916,0.987603,0.981520,0.984552,478,6,385,9,12.598179
YOLO + CNN,0.969248,0.961847,0.983573,0.972589,479,19,372,8,20.281638


In [46]:
comparison = pd.DataFrame(
    [
        yolo_results2,
        pipeline_results2

    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.965831,0.983087,0.954825,0.96875,465,8,383,22,11.632869
YOLO + CNN,0.922551,0.908382,0.956879,0.93200,466,47,344,21,18.474402


In [47]:
comparison = pd.DataFrame(
    [
        yolo_results3,
        pipeline_results3
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.958998,0.991285,0.934292,0.961945,455,4,387,32,12.135536
YOLO + CNN,0.941913,0.980176,0.913758,0.945802,445,9,382,42,19.545573


In [48]:
comparison = pd.DataFrame(
    [
        yolo_results4,
        pipeline_results4
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.818907,0.922680,0.735113,0.818286,358,30,361,129,11.631979
YOLO + CNN,0.733485,0.984674,0.527721,0.687166,257,4,387,230,16.764420


In [49]:
dangerous_yolo1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN")
]

In [50]:
dangerous_yolo2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN")
]

In [51]:
dangerous_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN")
]

In [52]:
dangerous_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN")
]

In [53]:
dangerous_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["pipeline"]=="OPEN")
]

In [54]:
dangerous_pipeline2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["pipeline"]=="OPEN")
]

In [55]:
dangerous_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["pipeline"]=="OPEN")
]

In [56]:
dangerous_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]

In [57]:
dangerous_both = df[
    (df["ground_truth"]=="CLOSE") & (df["yolo"]=="OPEN") & (df["pipeline"]=="OPEN")
]
dangerous_both1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") & (df_dark["yolo"]=="OPEN") & (df_dark["pipeline"]=="OPEN")
]
dangerous_both2 = df_night[
    (df_night["ground_truth"]=="CLOSE") & (df_night["yolo"]=="OPEN") & (df_night["pipeline"]=="OPEN")
]
dangerous_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") & (df_occlusion["yolo"]=="OPEN") & (df_occlusion["pipeline"]=="OPEN")
]
dangerous_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") & (df_motion_blur["yolo"]=="OPEN") & (df_motion_blur["pipeline"]=="OPEN")
]


In [58]:
comparison = pd.DataFrame(

    [   

        yolo_results,
        pipeline_results,
        yolo_results1,
        pipeline_results1,
        yolo_results2,
        pipeline_results2,
        yolo_results3,
        pipeline_results3,
        yolo_results4,
        pipeline_results4

    ],

    index=[

        "YOLO normal",
        "YOLO+CNN normal",
        "YOLO dark",
        "YOLO+CNN dark",
        "YOLO night",
        "YOLO+CNN night",
        "YOLO occlusion",
        "YOLO+CNN occlusion",
        "YOLO motion",
        "YOLO+CNN motion"

    ]

)

#latex_table = comparison.to_latex(index=True, float_format="{:.2f}".format)
#print("\\setlength{\\tabcolsep}{6pt}")
#print(latex_table)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO normal,0.988610,0.995842,0.983573,0.989669,479,2,389,8,14.287762
YOLO+CNN normal,0.984055,0.995807,0.975359,0.985477,475,2,389,12,21.382157
YOLO dark,0.982916,0.987603,0.981520,0.984552,478,6,385,9,12.598179
YOLO+CNN dark,0.969248,0.961847,0.983573,0.972589,479,19,372,8,20.281638
YOLO night,0.965831,0.983087,0.954825,0.968750,465,8,383,22,11.632869
YOLO+CNN night,0.922551,0.908382,0.956879,0.932000,466,47,344,21,18.474402
YOLO occlusion,0.958998,0.991285,0.934292,0.961945,455,4,387,32,12.135536
YOLO+CNN occlusion,0.941913,0.980176,0.913758,0.945802,445,9,382,42,19.545573
YOLO motion,0.818907,0.922680,0.735113,0.818286,358,30,361,129,11.631979
YOLO+CNN motion,0.733485,0.984674,0.527721,0.687166,257,4,387,230,16.764420


In [59]:
print()
print("Krytyczne błędy, wpuszczenie drapieżnika")
print("----------------")
print("YOLO dark:",len(dangerous_yolo1),"     YOLO night:",len(dangerous_yolo2),"     YOLO occlusion:",len(dangerous_yolo3),"    YOLO motion:",len(dangerous_yolo4))
print("YOLO+CNN dark:",len(dangerous_pipeline1),"YOLO+CNN night:",len(dangerous_pipeline2),"YOLO+CNN occlusion:",len(dangerous_pipeline3),"YOLO+CNN motion:",len(dangerous_pipeline4))
print("BOTH dark:",len(dangerous_both1),"     BOTH night:",len(dangerous_both2),"     BOTH occlusion:",len(dangerous_both3),"    BOTH motion:",len(dangerous_both4))


Krytyczne błędy, wpuszczenie drapieżnika
----------------
YOLO dark: 6      YOLO night: 8      YOLO occlusion: 4     YOLO motion: 30
YOLO+CNN dark: 19 YOLO+CNN night: 47 YOLO+CNN occlusion: 9 YOLO+CNN motion: 4
BOTH dark: 5      BOTH night: 6      BOTH occlusion: 2     BOTH motion: 2


In [60]:
locking_chicken_yolo = df[
    (df["ground_truth"]=="OPEN") & ((df["yolo"]=="CLOSE"))
]
locking_chicken_yolo1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE"))
]
locking_chicken_yolo2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE"))
]
locking_chicken_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE"))
]
locking_chicken_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE"))
]
locking_chicken_pipeline = df[
    (df["ground_truth"]=="OPEN") & ((df["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["pipeline"]=="CLOSE"))
]

locking_chicken_both = df[
    (df["ground_truth"]=="OPEN") & ((df["yolo"]=="CLOSE") | (df["pipeline"]=="CLOSE"))
]
locking_chicken_both1 = df_dark[
    (df_dark["ground_truth"]=="OPEN") & ((df_dark["yolo"]=="CLOSE") | (df_dark["pipeline"]=="CLOSE"))
]
locking_chicken_both2 = df_night[
    (df_night["ground_truth"]=="OPEN") & ((df_night["yolo"]=="CLOSE") | (df_night["pipeline"]=="CLOSE"))
]
locking_chicken_both3 = df_occlusion[
    (df_occlusion["ground_truth"]=="OPEN") & ((df_occlusion["yolo"]=="CLOSE") | (df_occlusion["pipeline"]=="CLOSE"))
]
locking_chicken_both4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="OPEN") & ((df_motion_blur["yolo"]=="CLOSE") | (df_motion_blur["pipeline"]=="CLOSE"))
]

In [61]:
print()
print("Niekrytyczne błędy, niewpuszczenie kur")
print("----------------")
print("YOLO dark:",len(locking_chicken_both1),"     YOLO night:",len(locking_chicken_both2),"     YOLO occlusion:",len(locking_chicken_both3),"    YOLO motion:",len(locking_chicken_both4))



Niekrytyczne błędy, niewpuszczenie kur
----------------
YOLO dark: 9      YOLO night: 25      YOLO occlusion: 42     YOLO motion: 234


In [62]:
data = [[len(dangerous_yolo), len(dangerous_yolo1), len(dangerous_yolo2), len(dangerous_yolo3), len(dangerous_yolo4)],
        [len(dangerous_pipeline), len(dangerous_pipeline1), len(dangerous_pipeline2), len(dangerous_pipeline3), len(dangerous_pipeline4)],
        [len(dangerous_both), len(dangerous_both1), len(dangerous_both2), len(dangerous_both3), len(dangerous_both4)]]
columns = ["normal", "dark", "night", "occlusion", "motion"]
index = ["yolo", "yolo+cnn", "both"]
table = pd.DataFrame(data, columns=columns, index=index)
tolatextable = table.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print("\\multicolumn{5}{c}{\\textbf{Krytyczne błędy, wpuszczenie przeciwnika}} ")
print(tolatextable)

\setlength{\tabcolsep}{6pt}
\multicolumn{5}{c}{\textbf{Krytyczne błędy, wpuszczenie przeciwnika}} 
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrr}
\toprule
 & normal & dark & night & occlusion & motion \\
\midrule
yolo & 2 & 6 & 8 & 4 & 30 \\
yolo+cnn & 2 & 19 & 47 & 9 & 4 \\
both & 1 & 5 & 6 & 2 & 2 \\
\bottomrule
\end{tabular}
\end{table}



In [63]:
data_locking = [[len(locking_chicken_yolo), len(locking_chicken_yolo1), len(locking_chicken_yolo2), len(locking_chicken_yolo3), len(locking_chicken_yolo4)],
    [len(locking_chicken_pipeline), len(locking_chicken_pipeline1), len(locking_chicken_pipeline2), len(locking_chicken_pipeline3), len(locking_chicken_pipeline4)],
    [len(locking_chicken_both), len(locking_chicken_both1), len(locking_chicken_both2), len(locking_chicken_both3), len(locking_chicken_both4)]]

columns_lock = ["normal", "dark", "night", "occlusion", "motion"]
index_lock = ["yolo", "yolo+cnn", "both"]

table2 = pd.DataFrame(data_locking, columns=columns, index=index)

tolatextable2 = table2.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print("\\multicolumn{5}{c}{\\textbf{Niekrytycznie błędy, niewpuszczenie kur}} ")
print(tolatextable2)

\setlength{\tabcolsep}{6pt}
\multicolumn{5}{c}{\textbf{Niekrytycznie błędy, niewpuszczenie kur}} 
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrr}
\toprule
 & normal & dark & night & occlusion & motion \\
\midrule
yolo & 8 & 9 & 22 & 32 & 129 \\
yolo+cnn & 12 & 8 & 21 & 42 & 230 \\
both & 13 & 9 & 25 & 42 & 234 \\
\bottomrule
\end{tabular}
\end{table}



In [64]:
latex_table = comparison.to_latex(index=True, float_format="{:.2f}".format,caption = "placeholder", label = "placeholder", position = "!h")
print("\\setlength{\\tabcolsep}{6pt}")
print(latex_table)

\setlength{\tabcolsep}{6pt}
\begin{table}[!h]
\caption{placeholder}
\label{placeholder}
\begin{tabular}{lrrrrrrrrr}
\toprule
 & Accuracy & Precision & Recall & F1 & TP & FP & TN & FN & Time \\
\midrule
YOLO normal & 0.99 & 1.00 & 0.98 & 0.99 & 479 & 2 & 389 & 8 & 14.29 \\
YOLO+CNN normal & 0.98 & 1.00 & 0.98 & 0.99 & 475 & 2 & 389 & 12 & 21.38 \\
YOLO dark & 0.98 & 0.99 & 0.98 & 0.98 & 478 & 6 & 385 & 9 & 12.60 \\
YOLO+CNN dark & 0.97 & 0.96 & 0.98 & 0.97 & 479 & 19 & 372 & 8 & 20.28 \\
YOLO night & 0.97 & 0.98 & 0.95 & 0.97 & 465 & 8 & 383 & 22 & 11.63 \\
YOLO+CNN night & 0.92 & 0.91 & 0.96 & 0.93 & 466 & 47 & 344 & 21 & 18.47 \\
YOLO occlusion & 0.96 & 0.99 & 0.93 & 0.96 & 455 & 4 & 387 & 32 & 12.14 \\
YOLO+CNN occlusion & 0.94 & 0.98 & 0.91 & 0.95 & 445 & 9 & 382 & 42 & 19.55 \\
YOLO motion & 0.82 & 0.92 & 0.74 & 0.82 & 358 & 30 & 361 & 129 & 11.63 \\
YOLO+CNN motion & 0.73 & 0.98 & 0.53 & 0.69 & 257 & 4 & 387 & 230 & 16.76 \\
\bottomrule
\end{tabular}
\end{table}



In [65]:
improved = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN") &
    (df_dark["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,13.5611,16.3327,[1],[0],[1]


In [66]:
improved = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN") &
    (df_night["pipeline"]=="CLOSE")

]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,13.3363,17.7676,[1],[0],[1]
840,raptor__gbif_raptor_00731_jpg.rf.DaBUw9KxmsnkZ...,CLOSE,OPEN,CLOSE,9.4280,14.3477,[1],[0],[1]


In [67]:
improved = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN") &
    (df_occlusion["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,12.9796,16.2465,[1],[0],[1]
339,Image-88-8f30f6.jpg,CLOSE,OPEN,CLOSE,10.1434,13.6080,[1],[0],[1]


In [68]:
improved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN") &
    (df_motion_blur["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
91,coyote__lila_AMMonitor_Camera_Traps_NEK-VB1_00...,CLOSE,OPEN,CLOSE,9.9779,15.8583,[1],[0],[1]
166,fox__gbif_fox_0058_jpg.rf.276TrYjTZ8l8fHW2cbDV...,CLOSE,OPEN,CLOSE,9.4584,15.3017,[1],[0],[1]
186,fox__gbif_fox_0454_jpg.rf.ET1jFfTjXFmpusKl9rJT...,CLOSE,OPEN,CLOSE,10.6315,16.7491,[1],[0],[1]
193,fox__gbif_fox_0540_jpg.rf.SviRhT8VBkdRDGFU1QO6...,CLOSE,OPEN,CLOSE,10.4557,15.8941,[1],[0],[1]
196,fox__gbif_fox_0615_jpg.rf.xGBhUyQrn1rpiysNE7WH...,CLOSE,OPEN,CLOSE,10.4338,16.1384,[1],[0],[1]
198,fox__gbif_fox_0722_jpg.rf.4EaY143B8mKPFC0058JJ...,CLOSE,OPEN,CLOSE,11.3465,16.0037,[1],[0],[1]
226,Image-111-ed55ed.jpg,CLOSE,OPEN,CLOSE,10.4293,14.3087,[1],[0],[1]
246,Image-28-e7f0ae.jpg,CLOSE,OPEN,CLOSE,10.4507,14.9594,"[1, 1]",[0],[1]
250,Image-31-1d307b.jpg,CLOSE,OPEN,CLOSE,11.2493,15.9947,[1],[0],[1]
252,Image-32-08b1fa.jpg,CLOSE,OPEN,CLOSE,9.6538,13.9825,[1],[0],[1]


In [69]:
deproved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]
deproved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
210,fox__lila_WCS_Camera_Traps_0312_jpg.rf.WM0NCxZ...,CLOSE,CLOSE,OPEN,10.0067,15.0115,[1],[1],[0]
874,raptor__raptor_027_jpg.rf.PzIAqRQVSQEonrCHFVSD...,CLOSE,CLOSE,OPEN,11.2000,16.8325,"[1, 1]",[1],[0]


In [70]:
empty_images0 = 0
false_detections0 = 0
for row in results_dark:

    if len(row["objects_gt"]) == 0:

        empty_images0 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections0 += 1

print("Puste obrazy:",empty_images0)
print("Fałszywe detekcje:",false_detections0)
if empty_images0>0:

    print(
        "Odsetek:",
        false_detections0/empty_images0
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [71]:
empty_images1 = 0
false_detections1 = 0

for row in results_night:

    if len(row["objects_gt"]) == 0:

        empty_images1 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections1 += 1

print("Puste obrazy:",empty_images1)
print("Fałszywe detekcje:",false_detections1)

if empty_images1>0:

    print(
        "Odsetek:",
        false_detections1/empty_images1
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [72]:
empty_images2 = 0
false_detections2 = 0

for row in results_occlusion:

    if len(row["objects_gt"]) == 0:

        empty_images2 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections2 += 1

print("Puste obrazy:",empty_images2)
print("Fałszywe detekcje:",false_detections2)

if empty_images2>0:

    print(
        "Odsetek:",
        false_detections2/empty_images2
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [73]:
empty_images3 = 0
false_detections3 = 0

for row in results_motion_blur:

    if len(row["objects_gt"]) == 0:

        empty_images3 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections3 += 1

print("Puste obrazy:",empty_images3)
print("Fałszywe detekcje:",false_detections3)

if empty_images3>0:

    print(
        "Odsetek:",
        false_detections3/empty_images3
    )

Puste obrazy: 37
Fałszywe detekcje: 4
Odsetek: 0.10810810810810811


In [74]:

#ZMIEŃ CONFIDENC POTEM NA 0,5 I PORÓWNAJ!!!!!!